# Vern Pivot Cross Statistics

Six themes of questions about Vern's ES pivot: the opening setup, crossing activity, confirmation, subsequent paths, market context and stability over time.

The primary population contains **875 pivot-matched RTH sessions**, from January 3, 2020 through September 18, 2026. Requested coverage through October 30, 2026 extends beyond the retained market evidence.

Each insight states its purpose, summarizes what the calculation shows, then presents a chart or table. These are descriptive point estimates. Historical pivot availability is unverified and eligibility is retrospective; trade fills, costs and profitability are untested.


### Working with this notebook

Choose this workspace's `.venv` kernel, then Restart Kernel and Run All. All study-specific filters, aggregations, comparisons, findings and visual generation are visible below. Notebook outputs use a new study work directory on every run.

The retained ranked and baseline ledgers are numerical inputs; the saved S01–S20 CSVs are used only for independent comparison after each calculation. The notebook does not load website report JSON. Only explicitly selected narrative and evidence outputs are exported to Narada; code and these editor instructions stay private.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gzip
import hashlib
import json
import os
import uuid
from abacus_bhishma.paths import load_workspace_paths

paths = load_workspace_paths()
WORKSPACE, DATA_ROOT = paths.workspace_root, paths.publication_root
STUDY_DIR = WORKSPACE / "studies/30_verniman_pivot_statistics_es"
lock_path = Path(os.environ.get("ABACUS_NOTEBOOK_INPUT_LOCK", str(STUDY_DIR / "notebook-v2.inputs.lock.json")))
input_lock = json.loads(lock_path.read_text())
if hashlib.sha256((WORKSPACE / "uv.lock").read_bytes()).hexdigest() != input_lock["environment_lock_sha256"]:
    raise ValueError("Review changed notebook environment pins before executing.")
sources = {}
for name, pin in input_lock["artifacts"].items():
    source = (DATA_ROOT if pin["root"] == "publication" else WORKSPACE) / pin["path"]
    if hashlib.sha256(source.read_bytes()).hexdigest() != pin["sha256"]:
        raise ValueError(f"Changed pinned input: {name}")
    sources[name] = source
baseline = json.loads(gzip.decompress(sources["ledger"].read_bytes()))
ranked = json.loads(gzip.decompress(sources["ranked_ledger"].read_bytes()))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
OUTPUT_DIR = Path(os.environ.get("ABACUS_NOTEBOOK_OUTPUT_DIR", str(WORKSPACE / "work/30_verniman_pivot_statistics_es" / (stamp + "_notebook_v2") / "outputs")))
if not OUTPUT_DIR.is_absolute() or not OUTPUT_DIR.resolve().is_relative_to(WORKSPACE / "work/30_verniman_pivot_statistics_es"):
    raise ValueError("Notebook outputs require a new study work directory.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / "work/.matplotlib-cache"))

INSIGHT_IDS = ['opening-configuration', 'opening-separation', 'vwap-frequency', 'pivot-frequency', 'count-distribution', 'price-ema-counts', 'pivot-confirmation', 'returned-break-depth', 'delayed-confirmation', 'forward-displacement', 'excursion-reach', 'ema-recross', 'crossing-order', 'pivot-retest', 'pivot-to-vwap', 'cloud-context', 'pivot-session-rate', 'first-cross-timing', 'section-persistence', 'recent-history', 'pivot-frequency-trend']


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
get_ipython().run_line_magic("matplotlib", "inline")

COHORT = "pivot_matched_rth"
daily = pd.DataFrame(baseline["daily"])
sections = pd.DataFrame(baseline["sections"])
price_events = pd.DataFrame(baseline["price_events"])
ema_events = pd.DataFrame(baseline["ema_events"])
ranked_daily = pd.DataFrame(ranked["daily"])
ranked_sections = pd.DataFrame(ranked["sections"])
ranked_outcomes = pd.DataFrame(ranked["outcomes"])
for frame in (daily, sections, price_events, ema_events, ranked_daily, ranked_sections, ranked_outcomes):
    frame["session_date"] = pd.to_datetime(frame["session_date"])
matched_daily = ranked_daily.loc[(ranked_daily["session_mode"] == "rth") & ranked_daily["cohorts"].map(lambda values: COHORT in values)].copy()
matched_sections = ranked_sections.loc[(ranked_sections["session_mode"] == "rth") & ranked_sections["cohorts"].map(lambda values: COHORT in values)].copy()
pivot_daily = matched_daily.loc[(matched_daily["reference"] == "pivot") & (matched_daily["series"] == "ema6")].copy()
vwap_daily = matched_daily.loc[(matched_daily["reference"] == "vwap") & (matched_daily["series"] == "ema6")].copy()
pivot_outcomes = ranked_outcomes.loc[(ranked_outcomes["session_mode"] == "rth") & (ranked_outcomes["reference"] == "pivot") & (ranked_outcomes["series"] == "ema6") & ranked_outcomes["cohorts"].map(lambda values: COHORT in values)].copy()
if len(daily) != 875 or daily["session_date"].duplicated().any():
    raise ValueError("Expected 875 unique eligible sessions, including zero-cross days.")
if matched_daily.duplicated(["session_date", "reference", "series"]).any() or len(matched_daily) != 4 * len(daily):
    raise ValueError("Expected common-day price/EMA6 counts at both reference lines.")
if sections.duplicated(["session_date", "section"]).any() or len(sections) != 3 * len(daily):
    raise ValueError("Expected three unique RTH section rows per eligible day.")
if price_events["event_id"].duplicated().any() or pivot_outcomes["outcome_id"].duplicated().any():
    raise ValueError("Event/outcome identities must remain unique.")
if matched_daily["crosses"].isna().any() or (matched_daily["crosses"] < 0).any():
    raise ValueError("Undefined counts cannot become observed zeros.")
plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":11,
    "axes.spines.top":False, "axes.spines.right":False, "axes.edgecolor":"#a6aaa6",
    "text.color":"#233a30", "axes.labelcolor":"#233a30", "figure.facecolor":"white",
    "axes.facecolor":"white", "figure.dpi":130, "savefig.facecolor":"white",
    "svg.hashsalt":"abacus-pivot-notebook-v2"})


In [ ]:
# Verification plumbing: the actual statistics are calculated in the insight cells.
references = {f"S{number:02}": pd.read_csv(sources[f"reference_s{number:02}"]) for number in range(1, 21)}
checks = []
def selection(reference="pivot", series="ema6", section="All sections", horizon=None, condition=None):
    names = {"ema6":"EMA6", "price":"Price", "cloud":"cloud"}
    text = f"Pivot-matched · RTH · {names[series]}/{reference} · {section} · Both directions · All crosses"
    if horizon:
        text += " · " + horizon
    if condition:
        text += " · " + condition
    return text

def check_saved(measure, metric, selected, actual, period="overall: all"):
    frame = references[measure]
    rows = frame.loc[(frame["period"] == period) & (frame["selection"] == selected) & (frame["metric"] == metric)]
    if len(rows) != 1:
        raise ValueError(f"Expected one exact saved row for {measure}: {metric}, {selected}, {period}")
    reference = rows.iloc[0]
    for key, value in actual.items():
        expected = reference[key]
        if value is None or pd.isna(value):
            if not pd.isna(expected):
                raise ValueError(f"Null mismatch: {measure}/{key}")
        elif key in {"coverage"}:
            if str(value) != str(expected):
                raise ValueError(f"Coverage mismatch: {measure}")
        elif not np.isclose(float(value), float(expected), rtol=1e-11, atol=1e-8):
            raise ValueError(f"Saved comparison failed: {measure}/{metric}/{key}: {value} != {expected}")
    checks.append({"insight_id":CURRENT_INSIGHT,"measure":measure,"metric":metric,"selection":selected,"period":period,
                   "columns":list(actual),"passed":True})


In [ ]:
# These helpers only format evidence already computed in the insight cells.
MIME = "application/vnd.abacus.insight+json"
def publish_table(frame, columns, notes, *, measure):
    view = frame[[column["key"] for column in columns]].copy()
    rows = json.loads(view.to_json(orient="records", double_precision=15))
    payload = {"kind":"table", "columns":columns, "rows":rows, "notes":notes,
               "source":{"measure_id":measure,"cohort":COHORT}}
    formats = {}
    for column in columns:
        key, kind, precision = column["key"], column.get("format"), column.get("precision", 2)
        if kind in ("number", "integer"):
            formats[key] = lambda x, p=precision: f"{x:,.{p}f}" if pd.notna(x) else "—"
        elif kind == "percent":
            formats[key] = lambda x, p=precision: f"{x:.{p}%}" if pd.notna(x) else "—"
    markup = view.to_html(index=False, border=0, classes="pivot-evidence", formatters=formats, na_rep="—")
    for column in columns:
        markup = markup.replace("<th>" + column["key"] + "</th>", "<th>" + column["label"] + "</th>")
    style = "<style>.pivot-evidence{border-collapse:collapse;color:#233a30;background:white;font-size:14px;width:100%}.pivot-evidence th{background:#355c4a;color:white;text-align:left;padding:12px}.pivot-evidence td{padding:12px;border-bottom:1px solid #d8ded8}.pivot-evidence tbody tr:nth-child(even){background:#f1f5f1}</style>"
    import html
    caption = "".join("<p style='font-size:13px;color:var(--vscode-descriptionForeground,#58645c)'>" + html.escape(note) + "</p>" for note in notes)
    display({"text/html":style + markup + caption, MIME:payload}, raw=True)
    view.to_csv(OUTPUT_DIR / (measure.lower() + "-" + str(len(checks)) + ".csv"), index=False)

def chart_axes(title, subtitle):
    fig, ax = plt.subplots(figsize=(10.4, 5.4))
    fig.subplots_adjust(left=.11, right=.98, bottom=.22, top=.79)
    fig.text(.11,.965,title,fontsize=16,fontweight="bold",ha="left",va="top")
    fig.text(.11,.902,subtitle,fontsize=10,color="#667269",ha="left",va="top")
    ax.grid(axis="y",color="#dfe6df",linewidth=.7)
    ax.set_axisbelow(True)
    return fig, ax

def publish_figure(fig, alt, caption, *, measure, identity):
    fig.text(.5,.025,"Descriptive point estimates · no new confidence intervals",fontsize=9,color="#667269",ha="center")
    fig.savefig(OUTPUT_DIR / (identity + ".png"), dpi=150)
    fig.savefig(OUTPUT_DIR / (identity + ".svg"))
    display(fig)
    display({MIME:{"kind":"figure","alt":alt,"caption":caption,
                   "source":{"measure_id":measure,"cohort":COHORT}}},raw=True)
    plt.close(fig)


## Opening context

Where does the session begin relative to the pivot, and how far apart are the reference levels?


### How does the opening configuration relate to crossing activity?


Compare the day's EMA6–pivot crossing count across price opening above both pivot and VWAP, below both, between the lines, or on an equal boundary. Opening states use the actual 09:30 snapshot.

All eligible sessions remain in the count, including days without crossings. These are conditional descriptions; source availability and full-day eligibility are retrospective.


In [ ]:
CURRENT_INSIGHT = 'opening-configuration'
opening = pivot_daily.groupby("opening_configuration", dropna=False).agg(
    sessions=("session_date","nunique"), crossings=("crosses","sum"), mean_crossings=("crosses","mean"))
opening = opening.reindex(["above_both","below_both","between","equal"]).dropna(subset=["sessions"])
for state, row in opening.iterrows():
    check_saved("S15","crosses [crossings/session]",selection(condition="opening_configuration="+state),
                {"value":row["mean_crossings"],"denominator":row["sessions"],"numerator":row["crossings"]})


In [ ]:
display(Markdown(
    f"**Findings summary.** Mean daily EMA6–pivot counts were **{opening.loc['above_both','mean_crossings']:.2f} on {int(opening.loc['above_both','sessions'])} above-both days**, {opening.loc['below_both','mean_crossings']:.2f} on {int(opening.loc['below_both','sessions'])} below-both days, and {opening.loc['between','mean_crossings']:.2f} on {int(opening.loc['between','sessions'])} between-line days. Crossing activity varied with the opening configuration; this does not establish a predictive advantage. The equal-boundary group contains only {int(opening.loc['equal','sessions'])} days and is sparse."
))


In [ ]:
fig, ax = chart_axes("Crossing activity by opening configuration", "Mean EMA6–pivot crossings per eligible RTH session · zeros included")
x = np.arange(len(opening))
ax.bar(x, opening["mean_crossings"], color="#355c4a", width=.58)
ax.set_xticks(x, ["Above both","Below both","Between","Equal boundary"])
ax.set(ylabel="Mean crossings per session",ylim=(0,3.1))
for at, (_, row) in zip(x, opening.iterrows()):
    ax.annotate(f"{row['mean_crossings']:.2f}\nn={int(row['sessions'])}",(at,row["mean_crossings"]),xytext=(0,9),textcoords="offset points",ha="center",fontsize=10)
publish_figure(fig,"Mean EMA6–pivot crossings by price opening configuration, with eligible-day counts.","The equal-boundary group is below the 30-day support threshold. Configurations use price, pivot and VWAP at 09:30.",measure="S15",identity="opening-configuration")


### How far apart are pivot and VWAP at the opening?


Measure the absolute separation of the daily pivot and contemporaneous Globex VWAP at 09:30, once in ES points and once in basis points of the pivot. The two columns are averages of daily measurements, not conversions of an aggregate mean.

The table retains the observed spread of separation without treating it as a selected entry threshold.


In [ ]:
CURRENT_INSIGHT = 'opening-separation'
separation_rows = []
for field, label, unit in [("opening_separation_points","Absolute separation","ES points"),("opening_separation_bps","Relative separation","Basis points")]:
    values = pivot_daily[field].dropna()
    separation_rows.append({"measure":label,"unit":unit,"sessions":len(values),"mean":values.mean(),"median":values.median(),"p75":values.quantile(.75)})
    check_saved("S17",field+" ["+("points" if unit=="ES points" else "basis points")+"]",selection(),
                {"value":values.mean(),"median":values.median(),"denominator":len(values)})
separation = pd.DataFrame(separation_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** Across **{len(pivot_daily):,} sessions**, mean opening separation was **{separation.loc[0,'mean']:.2f} ES points** and **{separation.loc[1,'mean']:.2f} basis points**. The point median was {separation.loc[0,'median']:.2f}, below its mean, showing an asymmetric retained distribution. Separation is context, not evidence that a wider gap creates a tradable opportunity."
))


In [ ]:
publish_table(separation, [{'key': 'measure', 'label': 'Measurement', 'format': 'text'}, {'key': 'unit', 'label': 'Unit', 'format': 'text'}, {'key': 'sessions', 'label': 'Sessions', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median', 'format': 'number', 'precision': 2}, {'key': 'p75', 'label': '75th percentile', 'format': 'number', 'precision': 2}], ['One 09:30 measurement per eligible day; absolute pivot–VWAP separation. No opening price translation or cross-contract adjustment.'], measure='S17')


## Crossing activity

How often do price and EMA6 cross the reference levels, including sessions without a crossing?


### How often does EMA6 cross VWAP on pivot-matched days?


Count actual completed three-minute EMA6 transitions through Globex VWAP during RTH on the same 875 pivot-matched sessions used throughout this notebook. Keep zero-cross days in the denominator.

This is a comparison line within the pivot study, with no pooling of the larger VWAP-only populations.


In [ ]:
CURRENT_INSIGHT = 'vwap-frequency'
vwap_frequency = vwap_daily["crosses"]
vwap_stats = pd.DataFrame([{"sessions":len(vwap_frequency),"crossings":vwap_frequency.sum(),"mean":vwap_frequency.mean(),"median":vwap_frequency.median(),"zero_days":int(vwap_frequency.eq(0).sum())}])
check_saved("S01","crosses [crossings/session]",selection(reference="vwap"),
            {"value":vwap_frequency.mean(),"median":vwap_frequency.median(),"denominator":len(vwap_frequency),"numerator":vwap_frequency.sum()})


In [ ]:
display(Markdown(
    f"**Findings summary.** EMA6 crossed VWAP **{int(vwap_frequency.sum()):,} times** across {len(vwap_frequency):,} matched sessions: a mean of **{vwap_frequency.mean():.2f} per session**, with median {vwap_frequency.median():.0f}. A total of {int(vwap_frequency.eq(0).sum())} sessions had no cross. These are recurrence counts within the pivot-matched cohort."
))


In [ ]:
publish_table(vwap_stats, [{'key': 'sessions', 'label': 'Sessions', 'format': 'integer', 'precision': 0}, {'key': 'crossings', 'label': 'Crossings', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean / session', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median', 'format': 'number', 'precision': 2}, {'key': 'zero_days', 'label': 'Zero-cross days', 'format': 'integer', 'precision': 0}], ['Actual EMA6–VWAP strict side changes, 09:30–16:00 New York. The opening state and VWAP reset are not crossings.'], measure='S01')


### How often does EMA6 cross the daily pivot?


Calculate daily EMA6–pivot crossing frequency for the full matched RTH sample and its up/down components. A crossing requires a completed strict-side transition; EMA6 merely being aligned with price is not a new crossing.

All 875 eligible sessions enter every daily mean, including zero-cross days.


In [ ]:
CURRENT_INSIGHT = 'pivot-frequency'
pivot_frequency_rows = []
for field, label, direction in [("crosses","All directions",None),("crosses_up","Upward","Up"),("crosses_down","Downward","Down")]:
    values = pivot_daily[field]
    pivot_frequency_rows.append({"direction":label,"sessions":len(values),"crossings":values.sum(),"mean":values.mean(),"median":values.median()})
    selected = selection() if direction is None else selection().replace("Both directions",direction.lower()+" direction")
    check_saved("S02","crosses [crossings/session]",selected,{"value":values.mean(),"median":values.median(),"denominator":len(values),"numerator":values.sum()})
pivot_frequency = pd.DataFrame(pivot_frequency_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** The sample contained **{int(pivot_frequency.loc[0,'crossings']):,} EMA6–pivot crossings**, a mean of **{pivot_frequency.loc[0,'mean']:.2f} per eligible session** and median {pivot_frequency.loc[0,'median']:.0f}. Upward and downward counts were {int(pivot_frequency.loc[1,'crossings']):,} and {int(pivot_frequency.loc[2,'crossings']):,}. Daily recurrence differs from the number of independent trading opportunities."
))


In [ ]:
publish_table(pivot_frequency, [{'key': 'direction', 'label': 'Crossing direction', 'format': 'text'}, {'key': 'sessions', 'label': 'Sessions', 'format': 'integer', 'precision': 0}, {'key': 'crossings', 'label': 'Crossings', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean / session', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median', 'format': 'number', 'precision': 2}], ['Equality carries the remembered strict side. Session opening initialization is not counted as a crossing.'], measure='S02')


### How common are zero-cross and repeated-cross days?


Place each day's actual EMA6 crossings into the frozen count categories 0, 1, 2, 3–4 and 5+. Compare pivot and VWAP within the same matched RTH sessions.

The no-cross category remains an observed result, and all categories use the same eligible-day denominator.


In [ ]:
CURRENT_INSIGHT = 'count-distribution'
def crossing_bucket(count):
    if pd.isna(count) or count < 0:
        return None
    return "0" if count == 0 else "1" if count == 1 else "2" if count == 2 else "3–4" if count < 5 else "5+"

categories = ["0","1","2","3–4","5+"]
count_rows = []
for reference, frame in [("pivot",pivot_daily),("vwap",vwap_daily)]:
    buckets = frame["crosses"].map(crossing_bucket)
    for category in categories:
        count = int(buckets.eq(category).sum())
        count_rows.append({"reference":reference,"category":category,"days":count,"share":count/len(frame)})
        check_saved("S03","count_bucket [fraction]",selection(reference=reference,condition="cross_count="+category),
                    {"share":count/len(frame),"numerator":count,"denominator":len(frame)})
count_distribution = pd.DataFrame(count_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** **{int(pivot_daily['crosses'].eq(0).sum())} of {len(pivot_daily)} sessions ({pivot_daily['crosses'].eq(0).mean():.1%}) had no EMA6–pivot crossing**, compared with {int(vwap_daily['crosses'].eq(0).sum())} ({vwap_daily['crosses'].eq(0).mean():.1%}) without an EMA6–VWAP crossing. The distribution shows how frequently these reference lines were crossed repeatedly, without treating multiple crosses within a day as independent days."
))


In [ ]:
fig, ax = chart_axes("Distribution of daily EMA6 crossing counts", "Share of the same 875 matched RTH sessions · frozen count categories")
x = np.arange(len(categories))
for offset, reference, color in [(-.18,"pivot","#355c4a"),(.18,"vwap","#a27b45")]:
    values = count_distribution.loc[count_distribution["reference"].eq(reference),"share"].to_numpy()
    ax.bar(x+offset,values,width=.34,color=color,label="Pivot" if reference=="pivot" else "VWAP")
ax.set_xticks(x,categories)
ax.set(xlabel="Crossings in a session",ylabel="Share of eligible sessions",ylim=(0,.7))
from matplotlib.ticker import PercentFormatter
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(frameon=False)
publish_figure(fig,"Shares of eligible days with 0, 1, 2, 3–4 or 5+ actual EMA6 crossings, comparing pivot and VWAP.","Each reference uses all 875 pivot-matched RTH sessions. Zero is an observed count, not missing support.",measure="S03",identity="count-distribution")


### How do price and EMA6 crossing counts differ?


Compare actual three-minute closing-price crossings with EMA6 line crossings at pivot and VWAP. Every series uses the same eligible sessions; differences are calculated within the common day/reference population.

These are recurrence counts, not returns from trading the corresponding events.


In [ ]:
CURRENT_INSIGHT = 'price-ema-counts'
paired_counts = matched_daily.groupby(["reference","series"]).agg(sessions=("session_date","nunique"),mean=("crosses","mean"),crossings=("crosses","sum")).reset_index()
for _, row in paired_counts.iterrows():
    check_saved("S05","crosses [crossings/session]",selection(reference=row["reference"],series=row["series"]),
                {"value":row["mean"],"denominator":row["sessions"],"numerator":row["crossings"]})
pivot_price_mean = paired_counts.loc[(paired_counts["reference"]=="pivot") & (paired_counts["series"]=="price"),"mean"].iloc[0]
pivot_ema_mean = paired_counts.loc[(paired_counts["reference"]=="pivot") & (paired_counts["series"]=="ema6"),"mean"].iloc[0]


In [ ]:
display(Markdown(
    f"**Findings summary.** Price averaged **{pivot_price_mean:.2f} pivot crossings per session**, compared with **{pivot_ema_mean:.2f} for EMA6** on the same {len(daily)} days. The smoothed indicator crossed less often in this sample. That count difference does not demonstrate that either event produces a superior trading result."
))


In [ ]:
fig, ax = chart_axes("Price and EMA6 crossings on the same sessions", "Mean actual crossings per eligible RTH session · common population")
x = np.arange(2)
for offset, series, color in [(-.18,"price","#a27b45"),(.18,"ema6","#355c4a")]:
    values = paired_counts.loc[paired_counts["series"].eq(series)].set_index("reference").loc[["pivot","vwap"],"mean"].to_numpy()
    bars=ax.bar(x+offset,values,width=.34,color=color,label="Price" if series=="price" else "EMA6")
    ax.bar_label(bars,fmt="%.2f",padding=5,fontsize=10)
ax.set_xticks(x,["Daily pivot","Globex VWAP"])
ax.set(ylabel="Mean crossings per session",ylim=(0,8))
ax.legend(frameon=False)
publish_figure(fig,"Mean price and EMA6 crossing counts at the pivot and VWAP on 875 common RTH sessions.","All series include zero-cross days. Both price and EMA6 require completed strict-side changes.",measure="S05",identity="price-ema-counts")


## Breaks & confirmation

Does an initially unconfirmed price break return or receive EMA6 confirmation, and what happens before that resolution?


### How often does a price break return before EMA6 confirms it?


Follow price breaks through Vern's pivot that begin with EMA6 opposite, equal with a remembered side, or not yet initialized. Does an actual EMA6 crossing occur before price returns through the pivot?

The percentage uses resolved episodes only. An episode with neither confirmation nor return remains censored. A confirmed episode can have its later return censored without losing its earlier confirmation.


In [ ]:
CURRENT_INSIGHT = 'pivot-confirmation'
def confirmation_outcome(row):
    if pd.notna(row.get("confirmation_at")):
        return "EMA6 confirmed before return"
    if pd.notna(row.get("returned_at")):
        return "Price returned before confirmation"
    return "Unresolved at observation end"

initially_unconfirmed = price_events.loc[price_events["ema_state_at_cross"].isin(["opposite","equal_with_remembered_side","unavailable_uninitialized"])].copy()
initially_unconfirmed["outcome"] = initially_unconfirmed.apply(confirmation_outcome,axis=1)
outcome_order = ["EMA6 confirmed before return","Price returned before confirmation","Unresolved at observation end"]
confirmation_table = initially_unconfirmed.groupby("outcome").agg(episodes=("event_id","size"),distinct_days=("session_date","nunique")).reindex(outcome_order)
resolved = initially_unconfirmed.loc[initially_unconfirmed["outcome"] != outcome_order[2]]
resolved_count, resolved_days = len(resolved), resolved["session_date"].nunique()
returned_count = int(confirmation_table.loc[outcome_order[1],"episodes"])
censored_count = int(confirmation_table.loc[outcome_order[2],"episodes"])
returned_share = returned_count / resolved_count
confirmation_table["resolved_share"] = confirmation_table["episodes"] / resolved_count
confirmation_table.loc[outcome_order[2],"resolved_share"] = np.nan
confirmation_table = confirmation_table.reset_index()
check_saved("S06","returned_before_confirmation [fraction]",selection(series="price"),
            {"share":returned_share,"numerator":returned_count,"denominator":resolved_count,"days":resolved_days,"censored":censored_count})


In [ ]:
display(Markdown(
    f"**Findings summary.** Of **{resolved_count:,} resolved price breaks** across {resolved_days:,} days, **{returned_count:,} ({returned_share:.1%}) returned through the pivot before EMA6 confirmation**. Another {censored_count:,} episodes remained unresolved and are excluded from that percentage. Almost half of these resolved, initially unconfirmed breaks returned first. This classification becomes known after the break and is not an entry-time signal."
))


In [ ]:
publish_table(confirmation_table, [{'key': 'outcome', 'label': 'Observed outcome', 'format': 'text'}, {'key': 'episodes', 'label': 'Episodes', 'format': 'integer', 'precision': 0}, {'key': 'resolved_share', 'label': 'Share of resolved episodes', 'format': 'percent', 'precision': 1}, {'key': 'distinct_days', 'label': 'Distinct days', 'format': 'integer', 'precision': 0}], ['Resolved denominator: 2,445 episodes; unresolved observations have no resolved-outcome percentage. Distinct-day counts overlap across rows.', 'Confirmation status is distinct from censoring of a later return.'], measure='S06')


### How deep are breaks that return without EMA6 confirmation?


Measure maximum directional penetration beyond the pivot during initially unconfirmed price episodes that return before an actual EMA6 confirmation. The population is selected by the subsequently observed return.

Depth is measured in unadjusted ES points; it is not an assumed loss, stop distance or execution price.


In [ ]:
CURRENT_INSIGHT = 'returned-break-depth'
returned_breaks = initially_unconfirmed.loc[initially_unconfirmed["outcome"] == "Price returned before confirmation"].copy()
depth_values = returned_breaks["episode_penetration_points"].dropna()
depth_table = pd.DataFrame([{"episodes":len(depth_values),"days":returned_breaks.loc[depth_values.index,"session_date"].nunique(),"mean":depth_values.mean(),"median":depth_values.median(),"p25":depth_values.quantile(.25),"p75":depth_values.quantile(.75)}])
check_saved("S07","returned_unconfirmed_depth [points]",selection(series="price"),
            {"value":depth_values.mean(),"median":depth_values.median(),"denominator":len(depth_values),"days":depth_table.loc[0,"days"]})


In [ ]:
display(Markdown(
    f"**Findings summary.** The **{len(depth_values):,} returned unconfirmed breaks** across {int(depth_table.loc[0,'days'])} days had mean penetration of **{depth_values.mean():.2f} ES points**, with median {depth_values.median():.2f}. The middle half lay between {depth_values.quantile(.25):.2f} and {depth_values.quantile(.75):.2f} points. Returned breaks can travel beyond the pivot before reversing; their eventual return is later information."
))


In [ ]:
publish_table(depth_table, [{'key': 'episodes', 'label': 'Returned episodes', 'format': 'integer', 'precision': 0}, {'key': 'days', 'label': 'Distinct days', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean points', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median points', 'format': 'number', 'precision': 2}, {'key': 'p25', 'label': '25th percentile', 'format': 'number', 'precision': 2}, {'key': 'p75', 'label': '75th percentile', 'format': 'number', 'precision': 2}], ['Conditional on return before confirmation. Unconfirmed unresolved episodes are not treated as returned breaks.'], measure='S07')


### How much time and movement precede delayed EMA6 confirmation?


Among price episodes with a later actual EMA6 crossing, measure elapsed time from the initial price break and directional price movement to confirmation. Same-close confirmations and already-aligned EMA6 states are distinct.

These measurements condition on eventual confirmation. They do not describe a filter available at the earlier price break.


In [ ]:
CURRENT_INSIGHT = 'delayed-confirmation'
delayed = price_events.loc[price_events["confirmation_kind"] == "delayed"].copy()
confirmation_rows = []
for key, label, unit, metric in [("confirmation_lag_minutes","Confirmation lag","Minutes","confirmation_delay [minutes]"),("price_movement_to_confirmation_points","Movement before confirmation","ES points","movement_before_confirmation [points]")]:
    valid = delayed.loc[delayed[key].notna()]
    values = valid[key]
    confirmation_rows.append({"measurement":label,"unit":unit,"episodes":len(values),"days":valid["session_date"].nunique(),"mean":values.mean(),"median":values.median()})
    check_saved("S08",metric,selection(series="price",condition="confirmation_kind=delayed"),
                {"value":values.mean(),"median":values.median(),"denominator":len(values),"days":valid["session_date"].nunique()})
delayed_table = pd.DataFrame(confirmation_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** Across **{len(delayed):,} delayed confirmations** on {delayed['session_date'].nunique()} days, the mean lag was **{delayed_table.loc[0,'mean']:.2f} minutes** and mean directional price movement before confirmation was **{delayed_table.loc[1,'mean']:.2f} ES points**. Waiting for confirmation changes the observation time and price reference; it cannot be retrospectively applied as though known at the original break."
))


In [ ]:
publish_table(delayed_table, [{'key': 'measurement', 'label': 'Measurement', 'format': 'text'}, {'key': 'unit', 'label': 'Unit', 'format': 'text'}, {'key': 'episodes', 'label': 'Episodes', 'format': 'integer', 'precision': 0}, {'key': 'days', 'label': 'Distinct days', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median', 'format': 'number', 'precision': 2}], ['Delayed confirmations only. The new EMA6 crossing must occur while the original price episode remains active.'], measure='S08')


## After the cross

What do complete subsequent price paths, EMA6 recrosses and pivot retests look like?


### What happens after an actual EMA6–pivot crossing?


Measure directional price displacement after an actual EMA6 crossing at complete 15-, 30- and 60-minute horizons. Positive means movement in the crossing direction; the event-close price is the reference.

Thirty minutes is the protocol's primary horizon. Paths extending beyond RTH or with missing evidence remain unavailable, and earlier extremes of the signal candle are excluded.


In [ ]:
CURRENT_INSIGHT = 'forward-displacement'
forward_rows = []
for horizon in ["15m","30m","60m"]:
    candidates = pivot_outcomes.loc[pivot_outcomes["horizon"] == horizon]
    valid = candidates.loc[candidates["valid"].eq(True) & candidates["displacement_points"].notna()]
    values = valid["displacement_points"]
    forward_rows.append({"horizon":horizon,"outcomes":len(valid),"days":valid["session_date"].nunique(),"unknown":len(candidates)-len(valid),"mean":values.mean(),"median":values.median(),"positive":values.gt(0).mean()})
    check_saved("S09","displacement [points]",selection(horizon=horizon),
                {"value":values.mean(),"median":values.median(),"denominator":len(valid),"days":valid["session_date"].nunique(),"unknown":len(candidates)-len(valid)})
    check_saved("S09","positive_movement [fraction]",selection(horizon=horizon),{"share":values.gt(0).mean(),"numerator":values.gt(0).sum(),"denominator":len(valid)})
forward_table = pd.DataFrame(forward_rows)
primary_paths = pivot_outcomes.loc[pivot_outcomes["horizon"].eq("30m") & pivot_outcomes["valid"].eq(True)].copy()
primary_candidates = pivot_outcomes.loc[pivot_outcomes["horizon"].eq("30m")].copy()


In [ ]:
display(Markdown(
    f"**Findings summary.** Across **{len(primary_paths):,} complete 30-minute paths** on {primary_paths['session_date'].nunique()} days, mean directional displacement was **{primary_paths['displacement_points'].mean():.2f} ES points** and {primary_paths['displacement_points'].gt(0).mean():.1%} ended strictly positive. Another {len(primary_candidates)-len(primary_paths)} candidate paths were unavailable. The selected crossing alone did not produce a positive average 30-minute directional displacement in this sample."
))


In [ ]:
publish_table(forward_table, [{'key': 'horizon', 'label': 'Horizon', 'format': 'text'}, {'key': 'outcomes', 'label': 'Complete paths', 'format': 'integer', 'precision': 0}, {'key': 'days', 'label': 'Distinct days', 'format': 'integer', 'precision': 0}, {'key': 'unknown', 'label': 'Unavailable', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean ES points', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median ES points', 'format': 'number', 'precision': 2}, {'key': 'positive', 'label': 'Strictly positive', 'format': 'percent', 'precision': 1}], ['Directional displacement starts at event close. Zero is distinct from positive; late/incomplete paths are excluded from each horizon denominator.', 'Multiple events on a day are dependent; these are point estimates without newly calculated confidence intervals.'], measure='S09')


### How far does price travel in either direction after the crossing?


For complete 30-minute paths, calculate favorable and adverse excursions from the crossing close and the share reaching 5, 10 or 20 ES points in each direction.

Both directions can be reached in the same path. Minute-bar extremes do not establish which threshold traded first, a fill price or net profitability.


In [ ]:
CURRENT_INSIGHT = 'excursion-reach'
reach_rows = []
for side in ["favorable","adverse"]:
    values = primary_paths[side+"_excursion_points"].dropna()
    check_saved("S10",side+"_excursion [points]",selection(horizon="30m"),
                {"value":values.mean(),"median":values.median(),"denominator":len(values)})
    for threshold in [5,10,20]:
        reached = primary_paths[f"{side}_reach_{threshold}"].dropna()
        reach_rows.append({"side":side,"threshold":threshold,"share":reached.mean(),"paths":len(reached)})
        check_saved("S10",f"{side}_reach_{threshold} [fraction]",selection(horizon="30m"),
                    {"share":reached.mean(),"numerator":reached.sum(),"denominator":len(reached)})
reach_table = pd.DataFrame(reach_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** The {len(primary_paths):,} complete paths had mean favorable excursion of **{primary_paths['favorable_excursion_points'].mean():.2f} ES points** and mean adverse excursion of **{primary_paths['adverse_excursion_points'].mean():.2f} points**. The share reaching 10 favorable points was {primary_paths['favorable_reach_10'].mean():.1%}, while {primary_paths['adverse_reach_10'].mean():.1%} reached 10 adverse points. Excursion opportunity and adverse exposure coexist; the chart does not encode threshold ordering or a trading result."
))


In [ ]:
fig, ax = chart_axes("Favorable and adverse threshold reach", "Complete 30-minute paths after actual EMA6–pivot crossings")
x = np.arange(3)
for offset, side, color in [(-.18,"favorable","#355c4a"),(.18,"adverse","#a27b45")]:
    values=reach_table.loc[reach_table["side"].eq(side),"share"].to_numpy()
    bars=ax.bar(x+offset,values,width=.34,color=color,label=side.capitalize())
    ax.bar_label(bars,labels=[f"{value:.1%}" for value in values],padding=5,fontsize=9)
ax.set_xticks(x,["5 ES points","10 ES points","20 ES points"])
ax.set(ylabel="Share of complete paths",ylim=(0,1.08))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(frameon=False)
publish_figure(fig,"Shares of complete 30-minute paths reaching favorable and adverse 5-, 10- or 20-point thresholds.","A path may reach both favorable and adverse thresholds. No intraminute order or execution is inferred.",measure="S10",identity="excursion-reach")


### How often does EMA6 avoid recrossing the pivot for 30 minutes?


Track the next reverse actual EMA6 crossing after each EMA6–pivot event. A complete 30-minute RTH window is required to classify absence of a recross.

Incomplete windows remain censored even when a reverse crossing is observed before the boundary. An EMA6 recross is distinct from a price return through the pivot.


In [ ]:
CURRENT_INSIGHT = 'ema-recross'
def no_recross_summary(frame):
    usable = frame.loc[frame["valid"].eq(True) & frame["no_recross"].notna()]
    return {"denominator":len(usable),"numerator":int(usable["no_recross"].eq(True).sum()),
            "days":usable["session_date"].nunique(),"censored":len(frame)-len(usable)}

recrosses = pd.DataFrame(ranked["recrosses"])
recrosses = recrosses.loc[recrosses["reference"].eq("pivot") & recrosses["session_mode"].eq("rth") & recrosses["horizon"].eq("30m") & recrosses["cohorts"].map(lambda values: COHORT in values)]
recross_stats = no_recross_summary(recrosses)
recross_share = recross_stats["numerator"] / recross_stats["denominator"]
recross_table = pd.DataFrame([
    {"outcome":"No reverse EMA6 crossing","windows":recross_stats["numerator"],"share":recross_share},
    {"outcome":"Reverse EMA6 crossing","windows":recross_stats["denominator"]-recross_stats["numerator"],"share":1-recross_share},
    {"outcome":"Censored/incomplete window","windows":recross_stats["censored"],"share":np.nan}])
check_saved("S11","no_recross [fraction]",selection(horizon="30m"),{**recross_stats,"share":recross_share})


In [ ]:
display(Markdown(
    f"**Findings summary.** No reverse EMA6 crossing occurred in **{recross_stats['numerator']:,} of {recross_stats['denominator']:,} complete windows ({recross_share:.1%})**, on {recross_stats['days']} supporting days. Another {recross_stats['censored']} windows were censored. Remaining on the same pivot side for 30 minutes does not establish a favorable price displacement or profitable trade."
))


In [ ]:
publish_table(recross_table, [{'key': 'outcome', 'label': 'Window outcome', 'format': 'text'}, {'key': 'windows', 'label': 'Windows', 'format': 'integer', 'precision': 0}, {'key': 'share', 'label': 'Share of complete windows', 'format': 'percent', 'precision': 1}], ['Only fully observed 30-minute RTH windows enter the probability denominator. Censored windows have no outcome percentage.'], measure='S11')


### Does follow-through differ between first and later crossings?


Compare complete 30-minute directional displacement after the first, second and third-or-later actual EMA6–pivot crossings of each RTH session.

Ordinals are known at the event close. Horizon availability remains specific to each event; a late crossing is not assigned a shorter window to complete the table.


In [ ]:
CURRENT_INSIGHT = 'crossing-order'
ordinal_rows = []
for ordinal in ["1","2","3+"]:
    candidates = primary_candidates.loc[primary_candidates["ordinal_group"].eq(ordinal)]
    valid = candidates.loc[candidates["valid"].eq(True)]
    values = valid["displacement_points"]
    ordinal_rows.append({"ordinal":ordinal,"outcomes":len(valid),"days":valid["session_date"].nunique(),"unknown":len(candidates)-len(valid),"mean":values.mean(),"median":values.median(),"positive":values.gt(0).mean()})
    check_saved("S13","displacement [points]",selection(horizon="30m",condition="crossing_ordinal="+ordinal),
                {"value":values.mean(),"median":values.median(),"denominator":len(valid),"days":valid["session_date"].nunique(),"unknown":len(candidates)-len(valid)})
ordinal_table = pd.DataFrame(ordinal_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** Mean 30-minute directional displacement was **{ordinal_table.loc[0,'mean']:.2f} ES points after first crossings** ({int(ordinal_table.loc[0,'outcomes'])} complete paths), {ordinal_table.loc[1,'mean']:.2f} after second crossings, and {ordinal_table.loc[2,'mean']:.2f} after third-or-later crossings. The table retains the different support and unavailable-path counts. These exploratory ordinal groups are not independent replications or a selected trading rule."
))


In [ ]:
publish_table(ordinal_table, [{'key': 'ordinal', 'label': 'Crossing ordinal', 'format': 'text'}, {'key': 'outcomes', 'label': 'Complete paths', 'format': 'integer', 'precision': 0}, {'key': 'days', 'label': 'Distinct days', 'format': 'integer', 'precision': 0}, {'key': 'unknown', 'label': 'Unavailable', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean ES points', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median ES points', 'format': 'number', 'precision': 2}, {'key': 'positive', 'label': 'Strictly positive', 'format': 'percent', 'precision': 1}], ['Ordinals reset by RTH session. Directional movement starts at each event close; 30-minute complete paths only.'], measure='S13')


### How often does price retest the pivot after an EMA6 crossing?


Look for the first subsequent completed three-minute bar containing the exact pivot within 30 minutes of an actual EMA6–pivot crossing. Complete windows determine retest incidence; observed touches separately support close classifications.

The touch definition uses bar ranges and does not infer an intrabar fill, a limit-order execution or the quality of a retest entry.


In [ ]:
CURRENT_INSIGHT = 'pivot-retest'
complete_retests = ema_events.loc[ema_events["retest_window_complete"].eq(True)].copy()
retest_count = int(complete_retests["retest_found"].eq(True).sum())
retest_share = retest_count / len(complete_retests)
retest_censored = len(ema_events)-len(complete_retests)
retest_table = pd.DataFrame([
    {"outcome":"Pivot retest observed","windows":retest_count,"share":retest_share},
    {"outcome":"No retest in complete window","windows":len(complete_retests)-retest_count,"share":1-retest_share},
    {"outcome":"Censored incidence window","windows":retest_censored,"share":np.nan}])
check_saved("S18","pivot_retest [fraction]",selection(horizon="30m"),
            {"share":retest_share,"numerator":retest_count,"denominator":len(complete_retests),"days":complete_retests["session_date"].nunique(),"censored":retest_censored})


In [ ]:
display(Markdown(
    f"**Findings summary.** A retest occurred in **{retest_count:,} of {len(complete_retests):,} complete windows ({retest_share:.1%})**, supported by {complete_retests['session_date'].nunique()} days. Another {retest_censored} incidence windows were censored. A pivot revisit is a frequent-path description under this bar-based definition; it does not measure whether a retest trade would have filled or made money."
))


In [ ]:
publish_table(retest_table, [{'key': 'outcome', 'label': 'Observed outcome', 'format': 'text'}, {'key': 'windows', 'label': 'Windows', 'format': 'integer', 'precision': 0}, {'key': 'share', 'label': 'Share of complete windows', 'format': 'percent', 'precision': 1}], ['Retest incidence requires the full declared 30-minute RTH window. An observed touch under a shorter window does not make that incidence window complete.'], measure='S18')


## VWAP & cloud

How do the other reference line and contemporaneous cloud states relate to pivot-cross paths?


### Does EMA6 reach VWAP before reversing through the pivot?


After an actual EMA6–pivot crossing, observe the next 30 minutes for a strict EMA6 crossing through VWAP before a reverse pivot crossing. The at-risk denominator contains resolved or fully observed sequences starting strictly before the other line.

Simultaneous crossings, already-beyond-VWAP starts and incomplete windows remain separate. A tie at a completed close has no inferred intrabar order.


In [ ]:
CURRENT_INSIGHT = 'pivot-to-vwap'
def sequence_summary(frame):
    resolved_categories = ["second_before_recross","recross_first","neither_within_30m"]
    resolved = frame.loc[frame["result"].isin(resolved_categories)]
    return {"denominator":len(resolved),"numerator":int(resolved["result"].eq("second_before_recross").sum()),
            "days":resolved["session_date"].nunique(),"censored":int(frame["censored"].eq(True).sum())}

sequences = pd.DataFrame(ranked["sequences"])
sequences = sequences.loc[sequences["reference"].eq("pivot") & sequences["first_reference"].eq("pivot") & sequences["session_mode"].eq("rth") & sequences["cohorts"].map(lambda values: COHORT in values)].copy()
sequence_stats = sequence_summary(sequences)
sequence_share = sequence_stats["numerator"] / sequence_stats["denominator"]
sequence_counts = sequences.groupby("result").size()
sequence_labels = {"second_before_recross":"VWAP crossed before pivot recross","recross_first":"Pivot recross occurred first","neither_within_30m":"Neither within complete 30 minutes","already_beyond_other":"Already beyond VWAP at start","simultaneous":"Simultaneous reference crossings","censored_no_endpoint":"Censored: no complete endpoint"}
sequence_table = pd.DataFrame([{"outcome":sequence_labels.get(key,key),"sequences":int(value),"share":value/sequence_stats["denominator"] if key in ["second_before_recross","recross_first","neither_within_30m"] else np.nan} for key,value in sequence_counts.items()])
check_saved("S14","second_before_recross [fraction]",selection(horizon="30m",condition="first_reference=pivot"),{**sequence_stats,"share":sequence_share})


In [ ]:
display(Markdown(
    f"**Findings summary.** EMA6 reached VWAP before recrossing the pivot in **{sequence_stats['numerator']:,} of {sequence_stats['denominator']:,} resolved or fully observed at-risk sequences ({sequence_share:.1%})**, supported by {sequence_stats['days']} days. Another {sequence_stats['censored']} sequences were censored. Excluded starting configurations remain visible below, so their large count is not silently absorbed into the conditional probability."
))


In [ ]:
publish_table(sequence_table, [{'key': 'outcome', 'label': 'Sequence disposition', 'format': 'text'}, {'key': 'sequences', 'label': 'Sequences', 'format': 'integer', 'precision': 0}, {'key': 'share', 'label': 'Share of at-risk resolved sequences', 'format': 'percent', 'precision': 1}], ['Only reach-before-recross, recross-first and neither-within-complete-window outcomes enter the conditional denominator. Other dispositions are shown with no conditional percentage.', 'Origins are pivot crosses; this table does not combine VWAP-origin paths.'], measure='S14')


### How do cloud states relate to movement after a pivot cross?


Compare complete 30-minute EMA6–pivot paths by contemporaneous cloud support, opposition or mixed state. Support requires directional EMA20/50 ordering and both native five-minute 15-minute slopes in the crossing direction; opposition requires the opposite configuration.

States are recorded at the event close. Conditional outcome groups are exploratory and share days with each other.


In [ ]:
CURRENT_INSIGHT = 'cloud-context'
cloud_rows = []
for state, candidates in primary_candidates.groupby("cloud_support", dropna=False):
    valid = candidates.loc[candidates["valid"].eq(True)]
    values = valid["displacement_points"]
    cloud_rows.append({"state":state,"outcomes":len(valid),"days":valid["session_date"].nunique(),"unknown":len(candidates)-len(valid),"mean":values.mean(),"median":values.median(),"positive":values.gt(0).mean()})
    check_saved("S16","displacement [points]",selection(horizon="30m",condition="cloud_support="+str(state)),
                {"value":values.mean(),"median":values.median(),"denominator":len(valid),"days":valid["session_date"].nunique(),"unknown":len(candidates)-len(valid)})
cloud_table = pd.DataFrame(cloud_rows).set_index("state").reindex(["support","opposition","mixed"]).reset_index()


In [ ]:
display(Markdown(
    f"**Findings summary.** Mean 30-minute directional displacement was **{cloud_table.loc[0,'mean']:.2f} ES points with cloud support** ({int(cloud_table.loc[0,'outcomes'])} paths), compared with **{cloud_table.loc[1,'mean']:.2f} points with opposition** ({int(cloud_table.loc[1,'outcomes'])} paths). Mixed states averaged {cloud_table.loc[2,'mean']:.2f} points. The observed ordering does not support assuming that the support label automatically improves this outcome; it is an exploratory conditional comparison."
))


In [ ]:
publish_table(cloud_table, [{'key': 'state', 'label': 'Cloud state', 'format': 'text'}, {'key': 'outcomes', 'label': 'Complete paths', 'format': 'integer', 'precision': 0}, {'key': 'days', 'label': 'Distinct days', 'format': 'integer', 'precision': 0}, {'key': 'unknown', 'label': 'Unavailable', 'format': 'integer', 'precision': 0}, {'key': 'mean', 'label': 'Mean ES points', 'format': 'number', 'precision': 2}, {'key': 'median', 'label': 'Median ES points', 'format': 'number', 'precision': 2}, {'key': 'positive', 'label': 'Strictly positive', 'format': 'percent', 'precision': 1}], ['Cloud state is contemporaneous with the EMA6 cross. Distinct-day support overlaps across states; events within a day are dependent.'], measure='S16')


## Timing & stability

When do crossings occur, which section states persist, and how stable are descriptive averages over time?


### When during RTH are EMA6–pivot crossings most frequent?


Compare the opening range (09:30–10:30), New York AM (10:30–12:30) and New York PM (12:30–16:00). These windows have unequal lengths, so normalize actual crossings by valid observed hours within each day and section, then average those daily rates.

Each section retains all eligible sessions, including zero-cross sessions. Section membership follows bar-open time; a cross becomes known at bar close.


In [ ]:
CURRENT_INSIGHT = 'pivot-session-rate'
def section_crossing_rates(frame):
    valid = frame["ema_observed_hours"].notna() & frame["ema_observed_hours"].gt(0) & frame["ema_crosses"].notna()
    values = frame.loc[valid].copy()
    values["crossings_per_hour"] = values["ema_crosses"] / values["ema_observed_hours"]
    result = values.groupby("section").agg(mean_per_hour=("crossings_per_hour","mean"),days=("session_date","nunique"),crossings=("ema_crosses","sum"),observed_hours=("ema_observed_hours","sum"))
    result["unavailable_day_sections"] = frame.groupby("section").size() - values.groupby("section").size().reindex(result.index,fill_value=0)
    return result

section_order = ["opening_range","new_york_am","new_york_pm"]
rates = section_crossing_rates(sections).reindex(section_order)
for key, label in [("opening_range","Opening range"),("new_york_am","AM"),("new_york_pm","PM")]:
    check_saved("S04","crosses_per_hour [crossings/hour]",selection(section=label),
                {"value":rates.loc[key,"mean_per_hour"],"denominator":rates.loc[key,"days"]})
if rates["unavailable_day_sections"].any() or not rates["days"].eq(len(daily)).all():
    raise ValueError("The retained section comparison must contain all eligible sessions.")


In [ ]:
display(Markdown(
    f"**Findings summary.** The mean daily rate was **{rates.loc['opening_range','mean_per_hour']:.2f} crossings per hour in the opening range**, compared with {rates.loc['new_york_am','mean_per_hour']:.2f} in New York AM and {rates.loc['new_york_pm','mean_per_hour']:.2f} in New York PM. After accounting for exposure, crossings were most frequent near the RTH open. This describes crossing activity, not trade quality."
))


In [ ]:
fig, ax = chart_axes("EMA6–pivot crossings are concentrated near the open", "Exposure-adjusted comparison · the same 875 sessions in every section")
x = np.arange(3)
ax.bar(x,rates["mean_per_hour"],color=["#355c4a","#65816e","#99ad9c"],width=.58)
for at, (_, row) in zip(x,rates.iterrows()):
    ax.annotate(f"{row['mean_per_hour']:.2f} / hour\nn={int(row['days'])} days",(at,row["mean_per_hour"]),xytext=(0,9),textcoords="offset points",ha="center",fontsize=10)
ax.set_xticks(x,["Opening range\n09:30–10:30","New York AM\n10:30–12:30","New York PM\n12:30–16:00"])
ax.set(ylabel="Mean daily crossings per observed hour",xlabel="New York RTH section",ylim=(0,1.15))
publish_figure(fig,"Mean daily EMA6–pivot rates are 0.87 per hour in the opening range, 0.32 in New York AM and 0.19 in New York PM.","Daily section rates are averaged; zero-cross sessions remain included. Window lengths are one, two and three-and-a-half hours.",measure="S04",identity="pivot-session-rate")


### When does the first EMA6–pivot crossing occur?


Locate each eligible session's first actual EMA6–pivot crossing by its completed-bar time. Show every half-hour bin chronologically and retain no-cross sessions as a separate category.

The timing mean conditions on days with a crossing; histogram shares use all eligible days. The final 15:30–16:00 bin includes the session-closing endpoint.


In [ ]:
CURRENT_INSIGHT = 'first-cross-timing'
first_minutes = pivot_daily["first_cross_minutes"].dropna()
no_cross_days = int(pivot_daily["crosses"].eq(0).sum())
first_counts, bin_edges = np.histogram(first_minutes.to_numpy(),bins=np.arange(0,391,30))
timing_counts = np.append(first_counts,no_cross_days)
timing_labels = [f"{(570+left)//60:02}:{(570+left)%60:02}" for left in bin_edges[:-1]] + ["No cross"]
if int(timing_counts.sum()) != len(pivot_daily):
    raise ValueError("Timing bins and no-cross category must cover every eligible session.")
check_saved("S12","first_cross_minutes [minutes]",selection(),{"value":first_minutes.mean(),"median":first_minutes.median(),"denominator":len(first_minutes),"unknown":no_cross_days})
for left, count in zip(bin_edges[:-1], first_counts):
    label = f"{left}–{left+30} inclusive" if left+30==390 else f"{left}–<{left+30}"
    check_saved("S12","first_cross_timing [fraction]",selection(condition="minutes_from_session_start="+label),
                {"share":count/len(pivot_daily),"numerator":count,"denominator":len(pivot_daily)})
check_saved("S12","first_cross_timing [fraction]",selection(condition="minutes_from_session_start=no_cross"),{"share":no_cross_days/len(pivot_daily),"numerator":no_cross_days,"denominator":len(pivot_daily)})


In [ ]:
display(Markdown(
    f"**Findings summary.** On **{len(first_minutes):,} days with a crossing**, the first cross occurred a mean **{first_minutes.mean():.1f} minutes after 09:30**, with median {first_minutes.median():.0f} minutes. Another {no_cross_days} sessions had no crossing. The full chronological distribution makes late first events and no-event days visible instead of conditioning them away."
))


In [ ]:
fig, ax = chart_axes("Time of the first EMA6–pivot crossing", "Each bar is a share of all 875 eligible RTH sessions")
x=np.arange(len(timing_counts))
ax.bar(x,timing_counts/len(pivot_daily),color=["#355c4a"]*(len(x)-1)+["#a27b45"],width=.7)
ax.set_xticks(x,timing_labels,rotation=45,ha="right",fontsize=9)
ax.set(ylabel="Share of eligible sessions",xlabel="Half-hour bin starting at New York time",ylim=(0,.46))
ax.yaxis.set_major_formatter(PercentFormatter(1))
fig.subplots_adjust(bottom=.26)
publish_figure(fig,"Chronological half-hour distribution of first actual EMA6–pivot crossing times, with no-cross sessions separate.","The final half-hour bin includes 16:00. Timing means use crossed days; these bars use all eligible sessions.",measure="S12",identity="first-cross-timing")


### Which cloud states persist from the opening range into later sections?


Compare cloud ordering at the opening-range endpoint with New York AM and PM endpoints. Each probability conditions on the observed opening-range source state and an observed destination.

These are endpoint-to-endpoint descriptions, not states known at the opening or causal forecasts of the next section.


In [ ]:
CURRENT_INSIGHT = 'section-persistence'
transitions = pd.DataFrame(ranked["transitions"])
cloud_transitions = transitions.loc[transitions["reference"].eq("pivot") & transitions["series"].eq("cloud") & transitions["metric"].eq("cloud_order") & transitions["from_section"].eq("opening_range") & transitions["cohorts"].map(lambda values: COHORT in values)].copy()
persistence_rows = []
for destination in ["new_york_am","new_york_pm"]:
    for state in ["bullish","bearish"]:
        candidates = cloud_transitions.loc[cloud_transitions["to_section"].eq(destination) & cloud_transitions["from_state"].eq(state)]
        valid = candidates.loc[candidates["valid"].eq(True) & candidates["persistent"].notna()]
        count = int(valid["persistent"].eq(True).sum())
        share = count/len(valid)
        persistence_rows.append({"source":state,"destination":"AM endpoint" if destination=="new_york_am" else "PM endpoint","days":len(valid),"persistent":count,"share":share,"unknown":len(candidates)-len(valid)})
        condition=f"section_state=opening_range→{destination}:cloud_order:{state}"
        check_saved("S19","state_persists [fraction]",selection(series="cloud",condition=condition),{"share":share,"numerator":count,"denominator":len(valid)})
persistence_table = pd.DataFrame(persistence_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** Bullish ordering persisted from the opening-range endpoint to the AM endpoint on **{persistence_table.loc[0,'share']:.1%} of {int(persistence_table.loc[0,'days'])} bullish-source days**; bearish ordering persisted on {persistence_table.loc[1,'share']:.1%} of {int(persistence_table.loc[1,'days'])} bearish-source days. The PM comparison uses its own destination endpoint. Persistence describes how observed section states relate and cannot be used as opening-time information."
))


In [ ]:
publish_table(persistence_table, [{'key': 'source', 'label': 'Opening-range state', 'format': 'text'}, {'key': 'destination', 'label': 'Compared with', 'format': 'text'}, {'key': 'days', 'label': 'Observed source days', 'format': 'integer', 'precision': 0}, {'key': 'persistent', 'label': 'Persistent days', 'format': 'integer', 'precision': 0}, {'key': 'share', 'label': 'Persistence share', 'format': 'percent', 'precision': 1}, {'key': 'unknown', 'label': 'Unavailable', 'format': 'integer', 'precision': 0}], ['Cloud state at the 10:30 opening-range endpoint is compared with the 12:30 AM and 16:00 PM endpoints. Each row uses its own observed source-state denominator.'], measure='S19')


### How does recent crossing activity compare with earlier history?


For EMA6–VWAP crossings on pivot-matched RTH days, compare the latest 20, 60 and 120 eligible observed sessions with the disjoint preceding history for each window.

The requested cutoff remains October 30, 2026, but available eligible evidence ends September 18. These are observed-session windows, not fixed calendar intervals; the 20-session estimate is sparse under the study's 30-day rule.


In [ ]:
CURRENT_INSIGHT = 'recent-history'
def disjoint_windows(frame, size):
    if frame["session_date"].duplicated().any():
        raise ValueError("Recent windows require unique eligible sessions.")
    ordered = frame.sort_values("session_date")
    if size <= 0 or size >= len(ordered):
        raise ValueError("Both recent and preceding windows need observed sessions.")
    return ordered.iloc[-size:].copy(), ordered.iloc[:-size].copy()

recent_rows = []
for size in [20,60,120]:
    recent, preceding = disjoint_windows(vwap_daily,size)
    recent_rows.append({"window":f"Last {size}","recent_n":len(recent),"recent_mean":recent["crosses"].mean(),"earlier_n":len(preceding),"earlier_mean":preceding["crosses"].mean(),"start":recent["session_date"].min().strftime("%Y-%m-%d"),"end":recent["session_date"].max().strftime("%Y-%m-%d")})
    for part, period in [(recent,f"recent: last_{size}"),(preceding,f"preceding: before_last_{size}")]:
        coverage=part["session_date"].min().strftime("%Y-%m-%d")+"–"+part["session_date"].max().strftime("%Y-%m-%d")
        check_saved("S20","crosses [crossings/session]",selection(reference="vwap",condition="source_statistic=S01"),
                    {"value":part["crosses"].mean(),"median":part["crosses"].median(),"denominator":len(part),"numerator":part["crosses"].sum(),"coverage":coverage},period=period)
recent_table = pd.DataFrame(recent_rows)


In [ ]:
display(Markdown(
    f"**Findings summary.** EMA6–VWAP crossings averaged **{recent_table.loc[0,'recent_mean']:.2f} per session over the last {int(recent_table.loc[0,'recent_n'])} eligible sessions**, compared with {recent_table.loc[0,'earlier_mean']:.2f} over the disjoint preceding {int(recent_table.loc[0,'earlier_n'])}. The recent window ended **{recent_table.loc[0,'end']}**. The longer windows show how sensitive the comparison is to the chosen observed-session horizon; these exploratory estimates do not establish a new market regime."
))


In [ ]:
publish_table(recent_table, [{'key': 'window', 'label': 'Observed-session window', 'format': 'text'}, {'key': 'recent_n', 'label': 'Recent sessions', 'format': 'integer', 'precision': 0}, {'key': 'recent_mean', 'label': 'Recent mean', 'format': 'number', 'precision': 2}, {'key': 'earlier_n', 'label': 'Earlier sessions', 'format': 'integer', 'precision': 0}, {'key': 'earlier_mean', 'label': 'Earlier mean', 'format': 'number', 'precision': 2}, {'key': 'start', 'label': 'Recent start', 'format': 'text'}, {'key': 'end', 'label': 'Recent end', 'format': 'text'}], ['Means are EMA6–VWAP crossings per pivot-matched RTH session. Earlier history excludes the corresponding recent window.', 'The latest usable date is September 18, 2026. The 20-session row is below the 30-day support threshold.'], measure='S20')


### How does EMA6–pivot crossing frequency vary by year?


Calculate mean actual EMA6–pivot crossings per eligible RTH session in each year, including zero-cross days. The connected points show descriptive annual averages, not a fitted regression.

Coverage differs across years: the 2022 sample begins May 18 and 2026 ends September 18. Each point is labeled with its eligible-session count; the line alone is not a significance test or evidence of a monotonic trend.


In [ ]:
CURRENT_INSIGHT = 'pivot-frequency-trend'
def annual_crossing_summary(frame):
    if frame["session_date"].duplicated().any():
        raise ValueError("Annual summaries require one row per unique eligible session.")
    return frame.assign(year=frame["session_date"].dt.year).groupby("year",sort=True).agg(
        sessions=("session_date","nunique"),crossings=("ema_crosses","sum"),
        mean_crossings=("ema_crosses","mean"),first_date=("session_date","min"),last_date=("session_date","max"))

annual = annual_crossing_summary(daily)
for year,row in annual.iterrows():
    coverage=row["first_date"].strftime("%Y-%m-%d")+"–"+row["last_date"].strftime("%Y-%m-%d")
    check_saved("S02","crosses [crossings/session]",selection(),
                {"value":row["mean_crossings"],"denominator":row["sessions"],"numerator":row["crossings"],"coverage":coverage},period=f"year: {year}")
high_year, low_year = int(annual["mean_crossings"].idxmax()), int(annual["mean_crossings"].idxmin())


In [ ]:
display(Markdown(
    f"**Findings summary.** Annual means ranged from **{annual['mean_crossings'].min():.2f} to {annual['mean_crossings'].max():.2f} EMA6–pivot crossings per eligible session**. The highest observed mean was in {high_year}; the lowest was in {low_year}. Frequency fluctuated across the retained years. Different source coverage and eligible-session counts limit direct year-to-year comparisons; the line does not establish a persistent trend."
))


In [ ]:
fig, ax = chart_axes("Annual EMA6 crossings of Vern's pivot", "875 pivot-matched RTH sessions · zero-cross days retained")
years=annual.index.to_numpy()
ax.plot(years,annual["mean_crossings"],color="#355c4a",linewidth=2.7,marker="o",markersize=6)
for year,row in annual.iterrows():
    ax.annotate(f"{row['mean_crossings']:.2f}\nn={int(row['sessions'])}",(year,row["mean_crossings"]),xytext=(0,12),textcoords="offset points",ha="center",fontsize=10)
ax.set_xticks(years,[str(year)+("*" if year in [2022,2026] else "") for year in years])
ax.set(ylabel="Mean crossings per eligible RTH session",xlabel="Session year · New York calendar",ylim=(0,3.25))
publish_figure(fig,"Annual EMA6–pivot crossing means range from 1.91 to 2.50 per eligible session during 2020–2026, with session counts labeled.","*2022 starts May 18; 2026 ends September 18. Connected annual point estimates have different eligible coverage; no fitted trend or new confidence intervals.",measure="S02",identity="pivot-frequency-trend")


## Scope and reproduction notes

The calculations use the selected run's retained day, section and event records. Source events, contract/source eligibility and feature availability are inherited frozen inputs; this notebook does not reconstruct vendor data or retrospectively establish when a pivot became available.

All charts and tables show descriptive point estimates. The earlier ranked reports retain their saved bootstrap intervals; this edition does not calculate new intervals or treat multiple events on a day as independent observations. Requested coverage ends October 30, 2026, while the latest eligible evidence ends September 18.

These questions do not model trade fills, costs or net profitability. VWAP-only RTH/full-Globex populations remain separately attributed companion evidence. The following private verification cell requires every published insight to reconcile with exact saved reference rows.


In [ ]:
expected_insights = set(INSIGHT_IDS)
covered = {row["insight_id"] for row in checks if row["passed"]}
if covered != expected_insights:
    raise ValueError(f"Incomplete saved-result reconciliation: {expected_insights-covered}")
verification = {"result":"passed","source_run_id":"20261031_ranked_v2",
    "publication_run_id":"20261108T035627Z_notebook_themes_v2","checks":checks,
    "insight_coverage":sorted(covered),"primary_measure_coverage":[f"S{i:02}" for i in range(1,21)],
    "scope":"Selected descriptive point estimates and support recalculated from retained ledgers; no new event reconstruction or confidence intervals."}
(OUTPUT_DIR / "saved-statistics-verification.json").write_text(json.dumps(verification,indent=2)+"\n")
display(Markdown(f"**Reproduction check:** all {len(checks)} saved-row comparisons passed, covering {len(covered)} insights and 20 primary measures."))
